# Atlas Raman — Stage 15B Spectral Features (ROI-PCA + SAM + DWT)

Reproduces the three spectral-pattern feature families introduced in Stage 15B of the Atlas Raman classification pipeline.

**Dataset:** 87 confocal Raman files, 7,122 QC-passed pixels, 987-bin wavenumber axis (preprocessed).  
**Classes:** STEC / Non-STEC / Salmonella / H2O  
**Goal:** Show how ROI-PCA, SAM, and DWT features are constructed and which ones survived MI-based feature selection in Stage 15F.

## How to Run

From the worktree root, create the required symlinks before executing this notebook:

```bash
ln -s /path/to/NomadX/data_cache data_cache
ln -s /path/to/NomadX/artifacts  artifacts
ln -s /path/to/NomadX/.venv      .venv
export OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1
.venv/bin/jupyter nbconvert --to notebook --execute --inplace \
    FINAL/notebooks/05_stage15b_spectral_features.ipynb \
    --ExecutePreprocessor.timeout=600
```

All paths below are relative to the worktree root so the notebook is self-contained once the symlinks exist.

## Method

Stage 15B adds three complementary spectral-pattern families on top of the named-band features from Stage 15A.

### ROI-PCA
The 987-bin spectrum is split into chemically meaningful regions:
- **LPS / fingerprint** 800–1200 cm⁻¹ (polysaccharide, phosphate, C-O)
- **Amide** 1500–1700 cm⁻¹ (Amide I/II, protein secondary structure)
- **CH-stretch** 2800–3050 cm⁻¹ (lipid acyl chains, fatty acids)

PCA is run independently within each region. The top-N principal component scores become features (`pca_chstretch_PC1/2/3`, etc.). Each PC captures a different axis of spectral variance within that chemical window, compressing hundreds of correlated bins into a small number of orthogonal scores.

### SAM (Spectral Angle Mapper)
For each pixel, compute the cosine angle between the pixel spectrum and the per-class or per-subclass mean spectrum (template). Smaller angle = more similar to that template. Templates are built separately for:
- Full spectrum: `sam_class_*`, `sam_sub_*`
- LPS region only (800–1200 cm⁻¹): `sam_lps_class_*`, `sam_lps_sub_*`

The LPS-region subclass templates (e.g., `sam_lps_sub_O121H19`) encode pixel-level resemblance to a specific *E. coli* serogroup's LPS chemistry — useful because LPS O-antigen structure differs markedly between serogroups.

### DWT (Discrete Wavelet Transform)
The full spectrum is decomposed with Daubechies db4 wavelets at 6 levels. Energy (`dwt_energy_L1..L6`) and spectral entropy (`dwt_entropy_L1..L6`) are computed per level. Low-level (L1) captures high-frequency noise; high-level (L6) captures broad spectral shape. Together these 12 features summarise multi-scale intensity variation without requiring peak assignments.

In [1]:
import os, sys
import warnings
warnings.filterwarnings('ignore')

# Ensure atlas module is importable from worktree root
REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), '..')) \
    if 'FINAL' in os.getcwd() else os.getcwd()
# Walk up until we find the atlas/ directory
candidate = os.getcwd()
for _ in range(4):
    if os.path.isdir(os.path.join(candidate, 'atlas')):
        REPO_ROOT = candidate
        break
    candidate = os.path.dirname(candidate)

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

DATA = os.path.join(REPO_ROOT, 'data_cache')
ARTIFACTS = os.path.join(REPO_ROOT, 'artifacts')

print(f'REPO_ROOT : {REPO_ROOT}')
print(f'DATA      : {DATA}')
print(f'ARTIFACTS : {ARTIFACTS}')

REPO_ROOT : /Users/devashishthapliyal/Documents/NomadX/.claude/worktrees/agent-a28638f912225da16
DATA      : /Users/devashishthapliyal/Documents/NomadX/.claude/worktrees/agent-a28638f912225da16/data_cache
ARTIFACTS : /Users/devashishthapliyal/Documents/NomadX/.claude/worktrees/agent-a28638f912225da16/artifacts


## Section 4 — Load Cached Spectral Features

In [2]:
import numpy as np
import pandas as pd

sf_df = pd.read_parquet(os.path.join(DATA, 'spectral_features.parquet'))
print(f'spectral_features.parquet shape: {sf_df.shape}')
print()
sf_df.head(3)

spectral_features.parquet shape: (7122, 51)



,dwt_energy_L1,dwt_entropy_L1,dwt_energy_L2,dwt_entropy_L2,dwt_energy_L3,dwt_entropy_L3,dwt_energy_L4,dwt_entropy_L4,dwt_energy_L5,dwt_entropy_L5,...,sam_lps_sub_83972,sam_lps_sub_ATCC25922,sam_lps_sub_Dublin,sam_lps_sub_H2O,sam_lps_sub_Heidelburg,sam_lps_sub_K-12,sam_lps_sub_O103H2,sam_lps_sub_O121H19,sam_lps_sub_O157H7,sam_lps_sub_Typhimurium
0,0.000096,4.427778,0.003160,2.773325,0.085549,1.963544,0.879136,0.989300,2.180325,1.459417,...,0.144187,0.092952,0.075296,0.059777,0.141974,0.078120,0.080006,0.117119,0.163165,0.282601
1,0.000106,4.307195,0.003393,3.084966,0.090735,2.124253,0.810086,1.006494,2.184682,1.469627,...,0.143376,0.093570,0.077112,0.057360,0.141084,0.078347,0.080972,0.116244,0.161391,0.280792
2,0.000148,4.338338,0.003584,2.768160,0.082357,2.115299,0.837935,0.974341,2.193503,1.455480,...,0.140400,0.089867,0.074361,0.056893,0.138253,0.074735,0.077260,0.113557,0.159632,0.279166


In [3]:
# Group columns by prefix
prefixes = ['pca_', 'sam_lps_', 'sam_class_', 'sam_sub_', 'dwt_', 'roi_']
for pfx in prefixes:
    cols = [c for c in sf_df.columns if c.startswith(pfx)]
    print(f'{pfx:<15} {len(cols):>3} columns  |  {cols[:4]}...' if len(cols) > 4 else f'{pfx:<15} {len(cols):>3} columns  |  {cols}')

pca_             11 columns  |  ['pca_lps_PC1', 'pca_lps_PC2', 'pca_lps_PC3', 'pca_lps_PC4']...
sam_lps_         14 columns  |  ['sam_lps_class_H2O', 'sam_lps_class_Non-STEC', 'sam_lps_class_STEC', 'sam_lps_class_Salmonella']...
sam_class_        4 columns  |  ['sam_class_H2O', 'sam_class_Non-STEC', 'sam_class_STEC', 'sam_class_Salmonella']
sam_sub_         10 columns  |  ['sam_sub_83972', 'sam_sub_ATCC25922', 'sam_sub_Dublin', 'sam_sub_H2O']...
dwt_             12 columns  |  ['dwt_energy_L1', 'dwt_entropy_L1', 'dwt_energy_L2', 'dwt_entropy_L2']...
roi_              0 columns  |  []


## Section 5 — Load Raw Spectra and Sample 200 Pixels

In [4]:
X_all   = np.load(os.path.join(DATA, 'spectra_array_preprocessed.npy'))
wn      = np.load(os.path.join(DATA, 'wavenumber_axis_preprocessed.npy'))
qc_mask = np.load(os.path.join(DATA, 'qc_mask.npy')).astype(bool)

# QC-passed spectra align with spectral_features.parquet row-for-row
X_qc = X_all[qc_mask]                                     # (7122, 987)

# Load pixel-level labels (spectra.parquet = one row per pixel in X_all order)
sp_df = pd.read_parquet(os.path.join(DATA, 'spectra.parquet'))
sp_qc = sp_df[qc_mask].reset_index(drop=True)

print(f'X_all  shape : {X_all.shape}')
print(f'X_qc   shape : {X_qc.shape}   (QC-passed pixels)')
print(f'wn     shape : {wn.shape}   range [{wn.min():.1f}, {wn.max():.1f}] cm⁻¹')
print(f'labels shape : {sp_qc.shape}')

# Sample 200 pixels reproducibly
rng   = np.random.default_rng(0)
idx   = rng.choice(len(X_qc), size=200, replace=False)
X_sub = X_qc[idx]
meta_sub = sp_qc.iloc[idx].reset_index(drop=True)

print(f'\nSampled {len(idx)} pixels')
print('Class distribution in sample:')
print(meta_sub['primary_class'].value_counts().to_string())

X_all  shape : (7999, 987)
X_qc   shape : (7122, 987)   (QC-passed pixels)
wn     shape : (987,)   range [400.4, 3049.2] cm⁻¹
labels shape : (7122, 6)

Sampled 200 pixels
Class distribution in sample:
primary_class
Salmonella    72
STEC          67
Non-STEC      37
H2O           24


## Section 6 — Refit ROI-PCA and Visualise CH-Stretch PCs

In [5]:
from atlas.spectral_features import fit_roi_pca, transform_roi_pca, DEFAULT_ROI_PCA

print('DEFAULT_ROI_PCA regions:')
for key, ((lo, hi), n_comp) in DEFAULT_ROI_PCA.items():
    n_bins = int(((wn >= lo) & (wn <= hi)).sum())
    print(f'  {key:<12} [{lo:.0f}–{hi:.0f} cm⁻¹]  {n_bins} bins  top-{n_comp} PCs')

fitted_pca = fit_roi_pca(X_sub, wn, regions=DEFAULT_ROI_PCA)

print('\nExplained variance ratio per region:')
for key, info in fitted_pca.items():
    if info['pca'] is not None:
        evr = info['pca'].explained_variance_ratio_
        cumulative = evr.cumsum()[-1]
        evr_str = ', '.join(f'{v:.3f}' for v in evr)
        print(f'  {key:<12} [{evr_str}]  cumulative={cumulative:.3f}')

DEFAULT_ROI_PCA regions:
  lps          [800–1200 cm⁻¹]  240 bins  top-5 PCs
  amide        [1500–1700 cm⁻¹]  120 bins  top-3 PCs
  chstretch    [2800–3050 cm⁻¹]  150 bins  top-3 PCs



Explained variance ratio per region:
  lps          [0.780, 0.172, 0.037, 0.004, 0.002]  cumulative=0.995
  amide        [0.961, 0.036, 0.002]  cumulative=0.999
  chstretch    [0.946, 0.053, 0.000]  cumulative=1.000


In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

pca_scores = transform_roi_pca(X_sub, wn, fitted_pca)

pc1 = pca_scores['pca_chstretch_PC1']
pc2 = pca_scores['pca_chstretch_PC2']

classes = meta_sub['primary_class'].values
class_colors = {'STEC': '#d62728', 'Non-STEC': '#1f77b4', 'Salmonella': '#2ca02c', 'H2O': '#9467bd'}

fig, ax = plt.subplots(figsize=(6, 5))
for cls in ['STEC', 'Non-STEC', 'Salmonella', 'H2O']:
    mask_cls = classes == cls
    if mask_cls.sum() > 0:
        ax.scatter(pc1[mask_cls], pc2[mask_cls], label=cls,
                   color=class_colors.get(cls, 'grey'), alpha=0.7, s=40, edgecolors='none')

ax.set_xlabel('CH-stretch PC1', fontsize=11)
ax.set_ylabel('CH-stretch PC2  [pca_chstretch_PC2]', fontsize=11)
ax.set_title('ROI-PCA: CH-stretch region (2800–3050 cm\u207b\u00b9)\n200-pixel sample', fontsize=11)
ax.legend(title='Class', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('chstretch_pca_scatter.png', dpi=100, bbox_inches='tight')
plt.show()
print('pca_chstretch_PC2 descriptive stats:')
print(pd.Series(pc2, name='pca_chstretch_PC2').describe().round(4).to_string())

pca_chstretch_PC2 descriptive stats:
count    200.0000
mean      -0.0000
std        0.2384
min       -0.4999
25%       -0.1680
50%       -0.0622
75%        0.1131
max        0.8521


## Section 7 — Refit SAM Templates and Visualise LPS-Region Heatmap

In [7]:
from atlas.spectral_features import fit_sam_templates, transform_sam, LPS_REGION_FOR_SAM

primary_sub = meta_sub['primary_class'].values
subclass_sub = meta_sub['subclass'].fillna('H2O').values

print(f'LPS_REGION_FOR_SAM: {LPS_REGION_FOR_SAM} cm\u207b\u00b9')
print(f'Unique primary classes in sample : {sorted(set(primary_sub))}')
print(f'Unique subclasses in sample      : {sorted(set(subclass_sub))}')

templates = fit_sam_templates(
    X_sub,
    y_primary=primary_sub,
    y_subclass=subclass_sub,
    wn=wn,
    region=LPS_REGION_FOR_SAM,
)

sam_scores = transform_sam(X_sub, templates, wn=wn)
print(f'\nSAM feature keys ({len(sam_scores)} total):')
for k in sorted(sam_scores.keys()):
    print(f'  {k}')

LPS_REGION_FOR_SAM: (800.0, 1200.0) cm⁻¹
Unique primary classes in sample : ['H2O', 'Non-STEC', 'STEC', 'Salmonella']
Unique subclasses in sample      : ['83972', 'ATCC25922', 'Dublin', 'H2O', 'Heidelburg', 'K-12', 'O103H2', 'O121H19', 'O157H7', 'Typhimurium']

SAM feature keys (28 total):
  sam_class_H2O
  sam_class_Non-STEC
  sam_class_STEC
  sam_class_Salmonella
  sam_lps_class_H2O
  sam_lps_class_Non-STEC
  sam_lps_class_STEC
  sam_lps_class_Salmonella
  sam_lps_sub_83972
  sam_lps_sub_ATCC25922
  sam_lps_sub_Dublin
  sam_lps_sub_H2O
  sam_lps_sub_Heidelburg
  sam_lps_sub_K-12
  sam_lps_sub_O103H2
  sam_lps_sub_O121H19
  sam_lps_sub_O157H7
  sam_lps_sub_Typhimurium
  sam_sub_83972
  sam_sub_ATCC25922
  sam_sub_Dublin
  sam_sub_H2O
  sam_sub_Heidelburg
  sam_sub_K-12
  sam_sub_O103H2
  sam_sub_O121H19
  sam_sub_O157H7
  sam_sub_Typhimurium


In [8]:
# Heatmap: select a few representative pixels (2 per class) and all subclass LPS templates
lps_sub_keys = sorted(k for k in sam_scores if k.startswith('sam_lps_sub_'))

# Pick 2 pixels per primary class
sample_idx = []
for cls in ['STEC', 'Non-STEC', 'Salmonella', 'H2O']:
    cls_mask = np.where(primary_sub == cls)[0]
    chosen = cls_mask[:2] if len(cls_mask) >= 2 else cls_mask
    sample_idx.extend(chosen.tolist())

heat_data = np.column_stack([sam_scores[k][sample_idx] for k in lps_sub_keys])
row_labels = [f'{primary_sub[i]} ({i})' for i in sample_idx]
col_labels = [k.replace('sam_lps_sub_', '') for k in lps_sub_keys]

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(heat_data, aspect='auto', cmap='RdYlGn_r', vmin=0)
ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, rotation=35, ha='right', fontsize=9)
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=8)
ax.set_title('SAM angles — LPS region (800–1200 cm\u207b\u00b9), subclass templates\n'
             'Lower value = smaller spectral angle = more similar to that subclass LPS pattern', fontsize=10)
plt.colorbar(im, ax=ax, label='Angle (rad)')
plt.tight_layout()
plt.savefig('sam_lps_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()
print('sam_lps_sub_O121H19 descriptive stats:')
print(pd.Series(sam_scores['sam_lps_sub_O121H19'], name='sam_lps_sub_O121H19').describe().round(4).to_string())

sam_lps_sub_O121H19 descriptive stats:
count    200.0000
mean       0.1890
std        0.2173
min        0.0285
25%        0.0789
50%        0.1013
75%        0.2303
max        1.3776


## Section 8 — DWT Features

In [9]:
from atlas.spectral_features import dwt_features

dwt_dict = dwt_features(X_sub, wavelet='db4', max_level=6)
dwt_df   = pd.DataFrame(dwt_dict)

print(f'DWT feature matrix shape: {dwt_df.shape}')
print(f'Features: {list(dwt_df.columns)}')
print()

top5_var = dwt_df.var().sort_values(ascending=False).head(5)
print('Top-5 DWT features by variance:')
print(top5_var.round(6).to_string())

DWT feature matrix shape: (200, 12)
Features: ['dwt_energy_L1', 'dwt_entropy_L1', 'dwt_energy_L2', 'dwt_entropy_L2', 'dwt_energy_L3', 'dwt_entropy_L3', 'dwt_energy_L4', 'dwt_entropy_L4', 'dwt_energy_L5', 'dwt_entropy_L5', 'dwt_energy_L6', 'dwt_entropy_L6']

Top-5 DWT features by variance:
dwt_energy_L6     12.183906
dwt_energy_L5      3.280958
dwt_energy_L4      1.618038
dwt_entropy_L1     0.325573
dwt_entropy_L2     0.249966


## Section 9 — Stage 15F MI Survival Check

In [10]:
import json

with open(os.path.join(ARTIFACTS, 'stage15f_metadata.json')) as f:
    meta15f = json.load(f)

selected_35 = meta15f['feature_columns']
print(f'Stage 15F selected {len(selected_35)} features total.')

# Filter to Stage 15B spectral feature families
spectral_prefixes = ('pca_', 'sam_', 'dwt_', 'roi_silent', 'roi_ch_stretch', 'roi_lps_chain')
spectral_survivors = [c for c in selected_35 if c.startswith(spectral_prefixes)]

print(f'\nStage 15B spectral features that survived MI selection ({len(spectral_survivors)}):')
for feat in spectral_survivors:
    tag = ''
    if feat in ('pca_chstretch_PC2', 'pca_chstretch_PC3', 'sam_lps_sub_O121H19'):
        tag = '  <-- PCA/SAM survivor'
    elif feat.startswith('roi_'):
        tag = '  <-- ROI moment survivor'
    print(f'  {feat}{tag}')

# Highlight the three key survivors
key_survivors = ['pca_chstretch_PC2', 'pca_chstretch_PC3', 'sam_lps_sub_O121H19']
print()
print('Key survivors confirmed in selected_35:')
for feat in key_survivors:
    status = 'YES' if feat in selected_35 else 'NO'
    print(f'  {feat:40s}  {status}')

# Features that did NOT survive (DWT, sam_class, sam_sub full-spectrum)
all_spectral_cached = [c for c in sf_df.columns if c.startswith(spectral_prefixes)]
dropped = [c for c in all_spectral_cached if c not in selected_35]
print(f'\nSpectral features dropped by MI ({len(dropped)}):')
print('  ' + ', '.join(dropped[:10]) + ('...' if len(dropped) > 10 else ''))

Stage 15F selected 35 features total.

Stage 15B spectral features that survived MI selection (9):
  roi_ch_stretch_std  <-- ROI moment survivor
  roi_silent_kurt  <-- ROI moment survivor
  roi_lps_chain_entropy  <-- ROI moment survivor
  roi_silent_skew  <-- ROI moment survivor
  roi_lps_chain_kurt  <-- ROI moment survivor
  roi_lps_chain_skew  <-- ROI moment survivor
  pca_chstretch_PC2  <-- PCA/SAM survivor
  pca_chstretch_PC3  <-- PCA/SAM survivor
  sam_lps_sub_O121H19  <-- PCA/SAM survivor

Key survivors confirmed in selected_35:
  pca_chstretch_PC2                         YES
  pca_chstretch_PC3                         YES
  sam_lps_sub_O121H19                       YES

Spectral features dropped by MI (48):
  dwt_energy_L1, dwt_entropy_L1, dwt_energy_L2, dwt_entropy_L2, dwt_energy_L3, dwt_entropy_L3, dwt_energy_L4, dwt_entropy_L4, dwt_energy_L5, dwt_entropy_L5...


## Section 10 — Summary

Stage 15B contributed **3 of the 35 MI-selected features** (`pca_chstretch_PC2`, `pca_chstretch_PC3`, `sam_lps_sub_O121H19`) plus several `roi_` statistical moments (std, skew, kurtosis, entropy) that also came from the Stage 15B ROI-moment pipeline.

**CH-stretch PCs** (`pca_chstretch_PC2/3`, 2800–3050 cm⁻¹): The CH-stretch region encodes lipid acyl-chain length and saturation together with protein backbone CH-stretch. PC2 and PC3 capture the secondary axes of variance after PC1 (which largely tracks overall intensity). These PCs discriminate classes because bacterial membrane lipid composition differs markedly between *E. coli* (shorter, more saturated acyl chains) and *Salmonella* (different LPS lipid A structure), and between STEC and Non-STEC strains at a subtler level.

**SAM LPS-region template** (`sam_lps_sub_O121H19`, 800–1200 cm⁻¹): The cosine angle between a pixel's fingerprint-region spectrum and the O121:H19 STEC serogroup mean template. Pixels that closely resemble O121:H19 LPS chemistry score a small angle; water or Salmonella pixels score large angles. This feature survived MI because the O121:H19 LPS pattern is sufficiently distinct from other subclasses in the 800–1200 cm⁻¹ window.

**DWT features**: All 12 DWT energy/entropy features were dropped by per-fold MI. Multi-scale energy captures global spectral shape but appears redundant with band-fit and ROI-moment features under the per-fold MI criterion.

The overall Stage 15F LOSO accuracy with the full 35-feature set was **0.436** (LogReg-L2), below the raw-spectrum PLS-DA baseline of **0.603**, confirming that engineered features alone do not outperform the baseline on this dataset.